# Notebook 3 — Conversational RAG

People rarely ask one perfect question. They chat:

> **User:** Tell me about the company cars.
> **User:** Can I use one for a weekend trip?

The second question only makes sense together with the first. This notebook shows why such
**follow-up questions** break the RAG pipeline from Notebooks 1 and 2, and two ways to fix
it.

**What you will learn**

1. Why follow-up questions break simple RAG.
2. How to rewrite a follow-up into a standalone question.
3. A conversational RAG chain with a chat history per user.
4. Agentic RAG: give the search to an agent as a tool.
5. When to use a chain and when to use an agent.

**Before you start**

```bash
ollama pull gpt-oss:20b
ollama pull nomic-embed-text
uv sync
```

Then choose the `.venv` Python kernel for this notebook. ⏱️ About 40 minutes.

## 1. Setup

We build the same index as at the end of Notebook 2: the handbook and the IT FAQ, split at
headings, in an in-memory vector store.

In [1]:
from operator import itemgetter
from pathlib import Path

from langchain.agents import create_agent
from langchain_core.documents import Document
from langchain_core.messages import AIMessage, HumanMessage
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables import RunnableLambda, RunnablePassthrough
from langchain_core.tools import tool
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_text_splitters import MarkdownHeaderTextSplitter, RecursiveCharacterTextSplitter
from langgraph.checkpoint.memory import InMemorySaver
from pydantic import BaseModel, Field

from roadmap import get_chat_model, get_embeddings
from roadmap.loaders import load_html, load_text

llm = get_chat_model()
DATA = Path("data")

documents = load_text(DATA / "handbook.md") + load_html(DATA / "it_faq.html")
header_splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=[("#", "title"), ("##", "section")], strip_headers=False
)
size_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)

chunks = []
for doc in documents:
    for part in header_splitter.split_text(doc.page_content):
        part.metadata = {**doc.metadata, **part.metadata}
        chunks.extend(size_splitter.split_documents([part]))

vector_store = InMemoryVectorStore.from_documents(chunks, get_embeddings())
retriever = vector_store.as_retriever(search_kwargs={"k": 3})


def cite(doc: Document) -> str:
    section = doc.metadata.get("section") or doc.metadata.get("title", "")
    return f"{doc.metadata['source']} > {section}"


def format_docs(docs: list[Document]) -> str:
    return "\n\n---\n\n".join(doc.page_content for doc in docs)


print(f"{len(chunks)} chunks from {len(documents)} documents")

24 chunks from 2 documents


## 2. The problem: follow-up questions

![Follow-up questions need the history](assets/followup_problem.svg)

The search only sees the text it gets. A follow-up like "Can I use one for a weekend trip?"
does not say what "one" is. Let's compare the search results for three follow-ups, first
as the user typed them, then written out in full:

In [2]:
FOLLOW_UPS = [
    # (earlier question, follow-up as typed, follow-up written out in full)
    (
        "Tell me about the company cars.",
        "Can I use one for a weekend trip?",
        "Can I use a company car for a private weekend trip?",
    ),
    (
        "I lost my work phone.",
        "How fast do I have to report it?",
        "How fast do I have to report a lost work phone?",
    ),
    (
        "How do I connect to the VPN?",
        "Do I need it for email?",
        "Do I need the VPN for email?",
    ),
]


def sections(question: str) -> str:
    return " | ".join(
        (doc.metadata.get("section") or "")[:28] for doc in retriever.invoke(question)
    )


for earlier, typed, full in FOLLOW_UPS:
    print(f"after: {earlier}")
    print(f"  as typed: {typed:<52} -> {sections(typed)}")
    print(f"  in full:  {full:<52} -> {sections(full)}\n")

after: Tell me about the company cars.
  as typed: Can I use one for a weekend trip?                    -> 3. Vacation | 6. Expenses and travel | 2. Remote work
  in full:  Can I use a company car for a private weekend trip?  -> 7. Company vehicles | 9. Mobile phone policy | 6. Expenses and travel

after: I lost my work phone.
  as typed: How fast do I have to report it?                     -> How do I report a phishing e | 8. Equipment and IT security | My laptop is slow or broken.
  in full:  How fast do I have to report a lost work phone?      -> 8. Equipment and IT security | 9. Mobile phone policy | My laptop is slow or broken.

after: How do I connect to the VPN?
  as typed: Do I need it for email?                              -> How do I connect to the VPN? | How do I report a phishing e | How do I get new software?
  in full:  Do I need the VPN for email?                         -> How do I connect to the VPN? | How do I report a phishing e | What is the guest Wi-Fi pass



- **Company cars:** the follow-up as typed does not find "7. Company vehicles" at all. The
  search matches "weekend trip" with vacation and travel instead.
- **Lost phone:** "report it" matches "How do I report a phishing email?" first; the right
  section ("8. Equipment and IT security") is only second.
- **VPN:** this one works by luck: the follow-up still contains the strong word "email".

A RAG app cannot rely on luck. The search needs a question that stands on its own.

## 3. Rewrite the follow-up into a standalone question

> **Question rewriting** (also called query rewriting): an LLM reads the chat history and
> turns the latest question into a **standalone question**, one that can be understood
> without the history.

The rewrite prompt must be clear that the model should **not** answer, only rewrite. We also
ask for **structured output** (Stage 1, Notebook 2): in a first version that used plain text,
the model once returned "Rewrite: ..." followed by the question twice. With a schema, the
result is always just the question.

In [3]:
class StandaloneQuestion(BaseModel):
    question: str = Field(description="the rewritten question that can be understood on its own")


rewrite_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "Rewrite the user's latest question so that it can be understood without the chat "
            "history: wherever it says 'it', 'one', or 'them', name the thing itself. "
            "Keep it short. Do not answer the question.",
        ),
        MessagesPlaceholder("history"),
        ("user", "{question}"),
    ]
)
rewrite_chain = (
    rewrite_prompt
    | llm.with_structured_output(StandaloneQuestion)
    | (lambda result: result.question)
)

for earlier, typed, _ in FOLLOW_UPS:
    history = [HumanMessage(content=earlier), AIMessage(content="(an answer about it)")]
    rewritten = rewrite_chain.invoke({"history": history, "question": typed})
    print(f"{typed:<36} -> {rewritten}")

Can I use one for a weekend trip?    -> Can I use a company car for a weekend trip?


How fast do I have to report it?     -> How fast do I have to report the lost work phone?


Do I need it for email?              -> Do I need the VPN for email?


The model resolved every "it" and "one". The wording of the rule matters: a first version
said "replace words such as 'it' with what they refer to", and in longer chats the model
sometimes left "it" in place. "Name the thing itself" worked every time in our tests.

Rewriting costs one extra model call (one to two seconds here), so we only do it when there
is a history to look at.

## 4. A conversational RAG chain

![Conversational RAG](assets/conversational_rag.svg)

The full chain has four steps:

1. **Rewrite** the question, if there is a history.
2. **Search** with the rewritten question.
3. **Answer** with a prompt that contains the history, the retrieved context, and the
   question. The history helps the model phrase the answer for this conversation.
4. **Save** the question and the answer in the history.

The answer prompt uses the clearer grounding rule from Notebook 2, plus one new rule: if a
question is unclear, ask back instead of guessing.

In [4]:
answer_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You answer questions from Brightleaf employees, using only the context below. "
            "The answer may be a yes or a no, or may need a small step of reasoning from the "
            "context. If the question is unclear, ask one short question to clarify instead "
            "of guessing. Only if the context does not cover the question, say: I don't know. "
            "Keep answers to two or three sentences.\n\nContext:\n{context}",
        ),
        MessagesPlaceholder("history"),
        ("user", "{question}"),
    ]
)


def search_query(inputs: dict) -> str:
    """Rewrite follow-ups; use the first question of a conversation as it is."""
    if not inputs["history"]:
        return inputs["question"]
    return rewrite_chain.invoke(inputs)


conversational_rag = (
    RunnablePassthrough.assign(search_query=RunnableLambda(search_query))
    | RunnablePassthrough.assign(docs=itemgetter("search_query") | retriever)
    | RunnablePassthrough.assign(context=lambda x: format_docs(x["docs"]))
    | RunnablePassthrough.assign(answer=answer_prompt | llm | StrOutputParser())
)

Each user gets their own history, stored under a **session id**. To keep prompts short we
only pass the last six messages (three turns), the "trimmed memory" idea from Stage 1,
Notebook 3.

In [5]:
sessions: dict[str, list] = {}


def chat(session_id: str, question: str) -> None:
    history = sessions.setdefault(session_id, [])
    result = conversational_rag.invoke({"question": question, "history": history[-6:]})
    history.extend([HumanMessage(content=question), AIMessage(content=result["answer"])])

    print(f"[{session_id}] {question}")
    if result["search_query"] != question:
        print(f"   searched: {result['search_query']}")
    print(f"   sources:  {' | '.join(cite(doc) for doc in result['docs'])}")
    print(f"   answer:   {result['answer']}\n")


chat("ada", "Tell me about the company cars.")
chat("ada", "Can I use one for a weekend trip?")
chat("ada", "Who pays if I get a parking ticket with it?")

[ada] Tell me about the company cars.
   sources:  handbook.md > 7. Company vehicles | handbook.md > 9. Mobile phone policy | handbook.md > 13. Contact
   answer:   Brightleaf has four shared electric cars for client visits. Book them through the car‑sharing app, keep them clean, and return them with at least 50 % charge. Eating (except water), smoking, vaping, or private trips are not allowed, and parking fines or speeding tickets are paid by the driver.



[ada] Can I use one for a weekend trip?
   searched: Can I use a Brightleaf company car for a weekend trip?
   sources:  handbook.md > 7. Company vehicles | it_faq.html > IT Help Desk FAQ | handbook.md > Brightleaf Employee Handbook
   answer:   No. Company vehicles may only be used for client visits; private weekend trips are not permitted.



[ada] Who pays if I get a parking ticket with it?
   searched: Who pays if I get a parking ticket with the company car?
   sources:  handbook.md > 7. Company vehicles | handbook.md > 9. Mobile phone policy | handbook.md > 13. Contact
   answer:   The driver pays for any parking fines or speeding tickets incurred while using a company vehicle.



Every follow-up was rewritten before the search ("searched: ..."), so the right section was
found first each time. Look at the second turn: the rewrite also added the company name, and
two general intro chunks came along as sources. Company names add noise to the search
(Notebook 1); section 5 shows the same effect with an agent. A new user starts with an empty history:

In [6]:
chat("ben", "Can I use one for a weekend trip?")

[ben] Can I use one for a weekend trip?
   sources:  handbook.md > 3. Vacation | handbook.md > 6. Expenses and travel | handbook.md > 2. Remote work
   answer:   Could you clarify what you mean by “one”? Are you asking if you can use a vacation day for a weekend trip?



Ben has no history, so there is nothing to rewrite, and "one" is unclear. Thanks to the
"ask back" rule, the model asks what Ben means instead of guessing. Without that rule, in our
tests it decided that "one" meant a vacation day and confidently answered a different
question. Sessions keep users apart, and a clarifying question is better than a wrong guess.

## 5. Agentic RAG: search as a tool

![Agentic RAG](assets/agentic_rag.svg)

The chain above always does the same thing: one rewrite, one search, one answer. An
**agent** (Stage 1, Notebook 3) can decide for itself:

- not to search at all, for small talk like "Thanks!",
- to search several times, once for each part of a question,
- and to write its own search queries, which also takes care of follow-ups.

> **Agentic RAG:** RAG where an agent decides when and how to search, by calling a search
> tool, instead of a fixed chain that always searches once.

The search becomes a tool. Its **description** tells the agent how to use it, and it
matters more than you might think. Let's build the same agent with two descriptions:

In [7]:
def make_search_tool(description: str):
    @tool("search_documents", description=description)
    def search_documents(query: str) -> str:
        return "\n\n".join(f"[{cite(doc)}]\n{doc.page_content}" for doc in retriever.invoke(query))

    return search_documents


AGENT_PROMPT = (
    "You help Brightleaf employees. For every question about company rules, benefits, or IT, "
    "call search_documents first, once for each part of the question. Answer only with facts "
    "from the search results. The answer may be a yes or a no, or need a small step of "
    "reasoning from the results. If the results do not cover the question, reply with exactly "
    "this sentence and nothing else: The documents do not answer this question. Keep answers "
    "to two or three sentences."
)


def make_agent(description: str):
    return create_agent(
        llm,
        tools=[make_search_tool(description)],
        system_prompt=AGENT_PROMPT,
        checkpointer=InMemorySaver(),
    )


def ask_agent(agent, text: str, thread_id: str) -> None:
    config = {"configurable": {"thread_id": thread_id}}
    messages = agent.invoke({"messages": [HumanMessage(content=text)]}, config)["messages"]
    last_question = max(i for i, m in enumerate(messages) if m.type == "human")
    new_messages = messages[last_question:]
    searches = [
        call["args"]["query"] for m in new_messages if m.type == "ai" for call in m.tool_calls
    ]
    print(f"USER:     {text}")
    print(f"searches: {searches}")
    print(f"answer:   {new_messages[-1].content}\n")


VAGUE = (
    "Search Brightleaf's employee handbook and IT help desk FAQ. "
    "Use it for every question about company rules, benefits, or IT."
)
FOCUSED = (
    "Search the company documents (employee handbook and IT help desk FAQ). Use a short "
    "query about one topic only, for example 'printer color page limit'. Do not put the "
    "company name or document names in the query."
)

question = "I lost my work phone. How fast do I have to report it?"
print("=== vague description")
ask_agent(make_agent(VAGUE), question, "test")
print("=== focused description")
ask_agent(make_agent(FOCUSED), question, "test")

=== vague description


USER:     I lost my work phone. How fast do I have to report it?
searches: ['lost work phone report time Brightleaf employee handbook IT help desk FAQ', 'lost work phone report time IT help desk FAQ Brightleaf']
answer:   The documents do not answer this question.

=== focused description


USER:     I lost my work phone. How fast do I have to report it?
searches: ['lost work phone report time']
answer:   You must report a lost or stolen work phone to the IT help desk within 24 hours.



With the vague description, the agent copies the company and document names into its query.
Those words match the general parts of the documents (Notebook 1 showed the same effect), the
right section is pushed out, and the agent cannot find the answer. With the focused
description, it writes a short query about the topic and finds "within 24 hours".

**Lesson:** for an agent, the tool description is part of the prompt. Say exactly how the
tool should be used.

Now a longer conversation with the focused agent. Watch how many searches each message
triggers:

In [8]:
agent = make_agent(FOCUSED)

ask_agent(agent, "How much is the learning budget, and do I need the VPN for email?", "ada")
ask_agent(agent, "Thanks, that helps!", "ada")
ask_agent(agent, "Tell me about the company cars.", "ada")
ask_agent(agent, "Can I use one for a weekend trip?", "ada")
ask_agent(agent, "What is the dress code?", "ada")

USER:     How much is the learning budget, and do I need the VPN for email?
searches: ['learning budget', 'VPN email']
answer:   The learning budget is 1,200 EUR per year.  
The VPN is not required for email or chat; it is only needed for the HR portal and finance tools when working outside the office.



USER:     Thanks, that helps!
searches: []
answer:   You're welcome!



USER:     Tell me about the company cars.
searches: ['company cars']
answer:   Brightleaf provides four shared electric cars for client visits. They must be booked through the car‑sharing app, and you may only use them for company business—private trips are prohibited. While in a company vehicle, eating and drinking are not allowed (except water), smoking or vaping is forbidden, and you must leave the car clean and charged to at least 50 % before returning it. Parking fines and speeding tickets are paid by the driver.



USER:     Can I use one for a weekend trip?
searches: []
answer:   No. Company vehicles may only be used for client visits or other business purposes; private weekend trips are not permitted.



USER:     What is the dress code?
searches: ['dress code']
answer:   The handbook does not specify a dress code.



**What to notice**

- The two-part question got **two** searches, one per topic.
- "Thanks" got **no** search.
- For the weekend-trip follow-up, the agent either wrote its own full query, or answered
  from the search result that was already in the conversation. Both are fine: the
  checkpointer keeps earlier tool results in the agent's memory.
- The dress code question gets one plain sentence, with no advice. Earlier versions of the
  agent prompt said "say you don't know" or "only say that the documents do not answer it",
  and the agent still added general advice ("dress professionally", "ask HR"). Asking for
  **one exact sentence** fixed that. The agent does not always copy the sentence word for
  word, but it no longer adds advice.

## 6. Chain or agent?

| | Conversational RAG chain | Agent with a search tool |
|---|---|---|
| Searches per message | always exactly one | none, one, or several: the model decides |
| Follow-up questions | a rewrite step | the agent writes its own queries |
| Speed and cost | predictable | varies with the number of model calls |
| Control | you decide every step | the model decides; needs a clear tool description and step limits |
| Good for | FAQ bots, simple Q&A over documents | questions with several parts, chats that mix small talk and lookups |

Start with the chain: it is easier to test and to explain. Move to an agent when users ask
questions that need several searches or several tools.

## 7. Summary

| Idea | What it means | In code |
|---|---|---|
| Follow-up question | Only makes sense with the earlier messages | "Can I use one ...?" |
| Question rewriting | Turn a follow-up into a standalone question first | `rewrite_prompt` with `MessagesPlaceholder("history")` |
| Conversational RAG | Rewrite, search, answer with history, save | `conversational_rag` chain + a history per session |
| Session | One user's conversation, kept apart from others | `sessions[session_id]` |
| Agentic RAG | An agent decides when and how to search | `create_agent(llm, tools=[search_documents])` |
| Tool description | Tells the agent how to write its queries | `@tool(..., description=...)` |

This is the end of the Stage 2 notebooks.

➡️ **Next:** the Stage 2 projects put these ideas into small apps:
[PDF Q&A bot](pdf-qa-bot/) and [Profile chat](profile-chat/).